## Model-based risk tiers vs. FICO tiers
My first risk tiers used simple FICO cutoffs. Here I group borrowers by the **model's predicted default probability** instead, using the same tier sizes as the FICO tiers so the comparison is fair. Everything below uses the 20% test set only.

In [ ]:
import numpy as np
import pandas as pd

# Test set with actual outcomes and the model's predicted default probability
test = X_test.copy()
test['defaulted'] = y_test.values
test['pd'] = predictions

# Same FICO rule as my SQL tiers
test['fico_tier'] = np.where(test['fico_range_low'] < 680, 'HIGH',
                    np.where(test['fico_range_low'] < 720, 'MEDIUM', 'LOW'))

# Model tiers with the SAME group sizes as the FICO tiers
high_share = (test['fico_tier'] == 'HIGH').mean()
low_share = (test['fico_tier'] == 'LOW').mean()
high_cut = test['pd'].quantile(1 - high_share)
low_cut = test['pd'].quantile(low_share)
test['model_tier'] = np.where(test['pd'] >= high_cut, 'HIGH',
                     np.where(test['pd'] <= low_cut, 'LOW', 'MEDIUM'))

def tier_summary(col):
    g = test.groupby(col)['defaulted'].agg(loans='size', default_rate='mean', defaults='sum')
    g['share_of_loans'] = g['loans'] / len(test)
    g['share_of_all_defaults'] = g['defaults'] / test['defaulted'].sum()
    return g.loc[['HIGH', 'MEDIUM', 'LOW']].round(4)

print('FICO tiers'); print(tier_summary('fico_tier'))
print('\nModel tiers'); print(tier_summary('model_tier'))

In [ ]:
# Which features drive default risk the most?
# Standardized coefficients: the effect of a one standard deviation change in each feature
coef = pd.Series(model.coef_[0], index=X_train.columns) * X_train.std()
print(coef.sort_values(key=abs, ascending=False).round(3))

In [ ]:
import matplotlib.pyplot as plt

tiers = ['HIGH', 'MEDIUM', 'LOW']
fico = test.groupby('fico_tier')['defaulted'].mean().loc[tiers] * 100
mod = test.groupby('model_tier')['defaulted'].mean().loc[tiers] * 100

x = np.arange(3); w = 0.36
fig, ax = plt.subplots(figsize=(9, 5.4))
b1 = ax.bar(x - w/2, fico, w, color='#b07d24', label='FICO rule (score cutoffs)')
b2 = ax.bar(x + w/2, mod, w, color='#b02a4c', label='My model (predicted default probability)')
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.5, f'{b.get_height():.1f}%', ha='center')
ax.set_xticks(x, ['High risk', 'Medium risk', 'Low risk'])
ax.set_ylabel('Actual default rate (%)')
ax.set_title('My model finds risky borrowers better than FICO alone')
ax.legend(frameon=False)
for s in ['top', 'right']: ax.spines[s].set_visible(False)
plt.tight_layout()
plt.savefig('model_vs_fico_tiers.png', dpi=200)
plt.show()